# Utilisation API KUBIOS 

In [ ]:
import pandas as pd
import os
import csv
import json
import requests
from pathlib import Path
import time
import re

In [ ]:
"""
Script d'envoi des intervalles RR à l'API Kubios pour le calcul
des métriques HRV.

Le script parcourt tous les fichiers CSV d'un dossier contenant
des intervalles RR, envoie les données à l'API Kubios, puis
enregistre les résultats de l'analyse au format JSON.
Kubios calcul les metriques en appliquant une fenêtre glissante, ainsi, le fichier JSON de sortie a une série de valeur pour chaque métrique.
Le scipt prend la moyenne de ces valeurs pour avoir qu'une seule par metrique dans le logiciel.
"""

## V2

In [108]:
"""
Paramètres : 

- folder_path : dossier contenant les fichiers CSV avec les intervalles RR
- output_folder : dossier dans lequel seront enregistrés les résultats
  des analyses Kubios
- access_token : jeton d'authentification pour accéder à l'API fourni par Kubios l'achat 
- api_key : clé API fournie par Kubios
- url_api : URL de l'API permettant de lancer l'analyse HRV
"""

folder_path = Path(r"C:\Users\judupont\Desktop\bb_rr_bloc_v2_kubios")      
output_folder = Path(r"C:\Users\judupont\Desktop\stat_bb_rr_v2")
access_token = "access_token"
api_key = "api_key"
url_api = "https:blablabla"


output_folder.mkdir(exist_ok=True)

# ==============================
# TRAITEMENT DES FICHIERS
# ==============================
for file_path in folder_path.glob("*.csv"):

    print(f"\n--- Traitement de {file_path.name} ---")

    try:
        # Lire le CSV
        df = pd.read_csv(file_path)

        # Vérifier que la colonne existe
        if "RR_ms" not in df.columns:
            print("❌ Colonne RR_ms introuvable")
            continue

        # Extraire les valeurs
        data_list = df["RR_ms"].dropna().astype(float).tolist()

        if len(data_list) < 250:
            print("❌ Pas assez de données RR")
            continue

        print(f"Nombre d'intervalles RR : {len(data_list)}")

        """
        Construction du payload JSON envoyé à l'API Kubios.

        - type : indique que les données sont des intervalles RR (RRI)
        - data : liste des intervalles RR
        - analysis : type d'analyse HRV demandée
        """
        
        payload = {
            "type": "RRI",
            "data": data_list,
            "analysis": {
                "type": "timevarying"
            }
        }


       # En-têtes HTTP pour authentification auprès de l'API.

        headers = {
            "Authorization": f"Bearer {access_token}",
            "X-Api-Key": api_key,
            "Content-Type": "application/json"
        }

        # Envoyer la requête
        response = requests.post(url_api, headers=headers, json=payload)

        print("Status code:", response.status_code)

        # Vérifier si la réponse de kubios est en format JSON
        if response.status_code == 200:
            try:
                result = response.json()
                print("✅ Analyse réussie")
            except:
                print("⚠ Réponse non JSON reçue")
                result = {"error": response.text}
        else:
            print("❌ Erreur API :", response.text)
            result = {"error": response.text}

        # Sauvegarder le résultat
        output_file = output_folder / f"{file_path.stem}_result.json"
        with open(output_file, "w", encoding="utf-8") as out_f:
            json.dump(result, out_f, indent=2)

        # Petite pause pour éviter rate limit
        time.sleep(1)

    except Exception as e:
        print("❌ Erreur générale :", e)

print("\nTraitement terminé ✅")


--- Traitement de 0101CAR_bloc1.csv ---
Nombre d'intervalles RR : 1524
Status code: 200
✅ Analyse réussie

--- Traitement de 0101CAR_bloc2.csv ---
Nombre d'intervalles RR : 3813
Status code: 200
✅ Analyse réussie

--- Traitement de 0101CAR_bloc3.csv ---
Nombre d'intervalles RR : 3575
Status code: 200
✅ Analyse réussie

--- Traitement de 0101EMS_bloc1.csv ---
Nombre d'intervalles RR : 1858
Status code: 200
✅ Analyse réussie

--- Traitement de 0101EMS_bloc2.csv ---
Nombre d'intervalles RR : 3170
Status code: 200
✅ Analyse réussie

--- Traitement de 0101EMS_bloc3.csv ---
Nombre d'intervalles RR : 3494
Status code: 200
✅ Analyse réussie

--- Traitement de 0102PCR_bloc1.csv ---
Nombre d'intervalles RR : 1678
Status code: 200
✅ Analyse réussie

--- Traitement de 0102PCR_bloc2.csv ---
Nombre d'intervalles RR : 1514
Status code: 200
✅ Analyse réussie

--- Traitement de 0102PCR_bloc3.csv ---
Nombre d'intervalles RR : 940
Status code: 200
✅ Analyse réussie

--- Traitement de 0103BPS_bloc1.csv -

In [1]:
json_folder_v2 = Path(r"C:\Users\judupont\Desktop\stat_bb_rr_v2")
df_global_v2_path = r"C:\Users\judupont\Desktop\df_global.txt"

metrics_to_extract = [
    "mean_rr", "sdnn", "mean_hr", "min_hr", "max_hr", "rmssd", "pnn50", "lf_power", "hf_power", "lf_hf_power"]

metrics_final_names = [ "Mean_RR_ms", "SDNN_ms", "Mean_HR_beats/min", "Min_HR_beats/min", "Max_HR_beats/min", "RMSSD_ms", "pNN50_%", "LF_power_ms2", "HF_power_ms2", "LF_HF_ratio"]

metrics_mapping = dict(zip(metrics_to_extract, metrics_final_names))

# --------------------------
# TRAITEMENT JSON V2
# --------------------------
candidates_v2 = {}

for file_path in json_folder_v2.glob("*.json"):
    with open(file_path, "r", encoding="utf-8") as f:
        try:
            data = json.load(f)
        except:
            continue
    if "analysis" not in data:
        continue
    analysis = data["analysis"]

    match = re.match(r"(\d{4}[A-Z]{3})_bloc(\d+)_result", file_path.stem)
    if not match:
        continue
    inclusion_number = match.group(1)
    bloc_number = match.group(2)

    if inclusion_number not in candidates_v2:
        candidates_v2[inclusion_number] = {"Numero_inclusion": inclusion_number, "effective_prc_blocs": []}

    for metric in metrics_to_extract:
        key_mean = f"{metrics_mapping[metric]}_bloc{bloc_number}"
        if metric in analysis:
            values = analysis[metric]
            if isinstance(values, list) and len(values) > 0:
                arr = np.array(values, dtype=float)
                arr_clean = arr[~np.isnan(arr)]
                candidates_v2[inclusion_number][key_mean] = np.nanmean(arr_clean) if len(arr_clean) > 0 else np.nan
            else:
                candidates_v2[inclusion_number][key_mean] = values
        else:
            candidates_v2[inclusion_number][key_mean] = np.nan

    if "effective_prc" in analysis:
        arr_eff = np.nan_to_num(np.array(analysis["effective_prc"], dtype=float), nan=0.0)
        candidates_v2[inclusion_number]["effective_prc_blocs"].append(np.mean(arr_eff))

# Calcul global effective_prc
for cand in candidates_v2.values():
    cand["effective_prc"] = np.mean(cand["effective_prc_blocs"]) if len(cand["effective_prc_blocs"]) > 0 else np.nan
    del cand["effective_prc_blocs"]

# DataFrame final
df_results_v2 = pd.DataFrame(list(candidates_v2.values()))
df_results_v2 = df_results_v2.sort_values(by="Numero_inclusion").reset_index(drop=True)

# Merge Condition + Feuille
df_global_v2 = pd.read_csv(df_global_v2_path, sep="\t")
df_global_small_v2 = df_global_v2[["Numero_inclusion", "Condition", "Feuille"]]
df_global_small_v2 = df_global_small_v2[df_global_small_v2["Numero_inclusion"].isin(df_results_v2["Numero_inclusion"])]
df_results_v2 = df_results_v2.merge(df_global_small_v2, on="Numero_inclusion", how="left")

cols = df_results_v2.columns.tolist()
for col_to_move in ["Condition", "Feuille"]:
    if col_to_move in cols:
        cols.remove(col_to_move)
        cols.insert(1, col_to_move)
df_results_v2 = df_results_v2[cols]

# Tri décroissant selon effective_prc
df_results_v2 = df_results_v2.sort_values(by="effective_prc", ascending=False).reset_index(drop=True)
df_results_v2

,Numero_inclusion,Feuille,Condition,Mean_RR_ms_bloc1,SDNN_ms_bloc1,Mean_HR_beats/min_bloc1,Min_HR_beats/min_bloc1,Max_HR_beats/min_bloc1,RMSSD_ms_bloc1,pNN50_%_bloc1,...,SDNN_ms_bloc3,Mean_HR_beats/min_bloc3,Min_HR_beats/min_bloc3,Max_HR_beats/min_bloc3,RMSSD_ms_bloc3,pNN50_%_bloc3,LF_power_ms2_bloc3,HF_power_ms2_bloc3,LF_HF_ratio_bloc3,effective_prc
0,0804GOR,SAINTE-MARGUERITE,Réduction de la menace,726.905537,28.191551,82.587414,71.288974,92.279146,14.459477,0.687427,...,29.710089,73.972367,64.959571,84.931310,16.732771,0.759581,672.063162,93.348999,7.523863,99.978017
1,0628DMR,LAVERAN,Réduction de la menace,811.016279,12.518072,74.003405,69.519272,80.181200,6.233576,0.081614,...,11.672952,71.583151,67.495912,78.117441,6.366650,0.000000,88.848154,12.827416,6.950121,99.975300
2,0905SLR,CGD,Réduction de la menace,832.659177,32.268365,72.069687,64.451710,81.528945,25.255556,4.739893,...,26.964104,68.960990,63.251336,76.531549,22.637828,3.708078,472.891073,216.306003,2.249200,99.970202
3,0403DCR,POITIERS,Réduction de la menace,628.640974,17.672437,95.477057,87.526315,106.775559,8.439303,0.000000,...,23.732443,85.315940,76.358944,95.768457,13.340275,0.148846,426.731340,102.263101,4.342951,99.970138
4,0408BCS,POITIERS,Standard,766.095142,24.587025,78.352673,70.401417,88.247601,14.994751,0.340930,...,27.426276,74.052273,66.730755,86.437715,17.690928,1.037072,488.389257,141.080829,3.696226,99.969861
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
123,0301GNR,CAEN,Réduction de la menace,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,33.114723,72.857569,67.172759,80.086792,45.608463,21.309277,215.150667,459.204173,0.925783,6.695767
124,0129APR,LPC,Réduction de la menace,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3.642060
125,0305LCR,CAEN,Réduction de la menace,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.271825
126,0343PAS,CAEN,Standard,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.212490


In [2]:
# Vérification des NaN
mask_nan = df_results_v2.isna().any(axis=1)
lignes_nan = df_results_v2[mask_nan]

# Liste des candidats avec au moins un NaN
candidats_nan = lignes_nan['Numero_inclusion'].tolist()

if candidats_nan:
    print("⚠️ Candidats contenant au moins un NaN :")
    for c in candidats_nan:
        print(f"• {c}")
else:
    print("✅ Aucun NaN détecté dans le DataFrame.")

#  Supprimer les lignes avec NaN
df_results_v2 = df_results_v2[~mask_nan].reset_index(drop=True)
print(f"\n🎯 Nombre final de candidats conservés : {len(df_results_v2)}")

⚠️ Candidats contenant au moins un NaN :
• 0333MMS
• 0341LJS
• 0119LJS
• 0112DRR
• 0313FPR
• 0102PCR
• 0339NPR
• 0342VNS
• 0337BFS
• 0301GNR
• 0129APR
• 0305LCR
• 0343PAS
• 0117MSR

🎯 Nombre final de candidats conservés : 114


In [3]:
df_results_v2 = df_results_v2.round(2)
# --------------------------
# Affichage final
# --------------------------
pd.set_option('display.max_columns', None)
df_results_v2

,Numero_inclusion,Feuille,Condition,Mean_RR_ms_bloc1,SDNN_ms_bloc1,Mean_HR_beats/min_bloc1,Min_HR_beats/min_bloc1,Max_HR_beats/min_bloc1,RMSSD_ms_bloc1,pNN50_%_bloc1,LF_power_ms2_bloc1,HF_power_ms2_bloc1,LF_HF_ratio_bloc1,Mean_RR_ms_bloc2,SDNN_ms_bloc2,Mean_HR_beats/min_bloc2,Min_HR_beats/min_bloc2,Max_HR_beats/min_bloc2,RMSSD_ms_bloc2,pNN50_%_bloc2,LF_power_ms2_bloc2,HF_power_ms2_bloc2,LF_HF_ratio_bloc2,Mean_RR_ms_bloc3,SDNN_ms_bloc3,Mean_HR_beats/min_bloc3,Min_HR_beats/min_bloc3,Max_HR_beats/min_bloc3,RMSSD_ms_bloc3,pNN50_%_bloc3,LF_power_ms2_bloc3,HF_power_ms2_bloc3,LF_HF_ratio_bloc3,effective_prc
0,0804GOR,SAINTE-MARGUERITE,Réduction de la menace,726.91,28.19,82.59,71.29,92.28,14.46,0.69,731.05,63.36,11.50,778.73,27.74,77.08,67.68,88.25,14.59,0.61,635.19,72.74,8.89,811.38,29.71,73.97,64.96,84.93,16.73,0.76,672.06,93.35,7.52,99.98
1,0628DMR,LAVERAN,Réduction de la menace,811.02,12.52,74.00,69.52,80.18,6.23,0.08,112.01,11.14,10.56,826.99,11.17,72.58,68.10,79.98,6.17,0.00,95.47,11.97,7.69,838.35,11.67,71.58,67.50,78.12,6.37,0.00,88.85,12.83,6.95,99.98
2,0905SLR,CGD,Réduction de la menace,832.66,32.27,72.07,64.45,81.53,25.26,4.74,699.91,294.43,2.44,865.28,27.80,69.39,63.17,78.28,21.92,3.92,551.81,207.51,2.65,870.74,26.96,68.96,63.25,76.53,22.64,3.71,472.89,216.31,2.25,99.97
3,0403DCR,POITIERS,Réduction de la menace,628.64,17.67,95.48,87.53,106.78,8.44,0.00,226.72,48.58,4.88,666.74,21.43,90.01,79.32,102.95,11.41,0.13,372.68,75.84,4.74,703.64,23.73,85.32,76.36,95.77,13.34,0.15,426.73,102.26,4.34,99.97
4,0408BCS,POITIERS,Standard,766.10,24.59,78.35,70.40,88.25,14.99,0.34,453.26,89.33,5.02,778.98,24.83,77.03,68.30,89.06,15.95,0.50,438.56,116.30,3.78,810.38,27.43,74.05,66.73,86.44,17.69,1.04,488.39,141.08,3.70,99.97
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
109,0117POS,APHM,Standard,680.24,10.30,88.21,82.59,92.85,5.83,0.00,48.40,6.99,7.42,759.21,10.77,79.06,73.81,83.48,9.14,0.38,64.57,26.04,2.74,812.68,17.90,73.84,69.49,79.10,22.17,7.01,109.25,172.25,1.39,69.64
110,0331GRS,CAEN,Standard,933.96,58.16,64.31,53.80,76.16,57.26,17.61,1162.34,704.89,1.67,918.09,42.55,65.36,56.51,70.37,38.95,8.05,1183.60,522.93,2.94,904.37,29.15,66.41,58.07,74.41,27.29,4.59,462.74,262.85,2.32,69.24
111,0104IBS,LPC,Standard,811.71,25.31,73.94,64.68,86.12,17.87,1.87,347.94,78.61,4.51,904.03,22.78,66.39,59.82,72.42,18.46,2.02,228.60,92.28,3.01,938.31,27.46,63.97,57.85,71.02,17.89,2.00,412.84,62.84,6.71,61.01
112,0312JMS,CAEN,Standard,710.50,18.81,84.50,77.02,94.19,13.28,1.62,162.38,54.22,3.12,764.64,27.06,78.53,72.02,84.99,27.19,4.28,239.59,552.25,4.23,736.99,29.39,81.52,72.47,92.73,30.25,4.65,180.68,169.37,2.48,47.85


## V4

In [6]:
folder_path = Path(r"C:\Users\judupont\Desktop\bb_rr_bloc_v4_kubios")      
output_folder = Path(r"C:\Users\judupont\Desktop\stat_bb_rr_v4")
access_token = "access_token"
api_key = "api_key"
url_api = "https:blablabla"


output_folder.mkdir(exist_ok=True)

# ==============================
# TRAITEMENT DES FICHIERS
# ==============================

for file_path in folder_path.glob("*.csv"):

    print(f"\n--- Traitement de {file_path.name} ---")

    try:
        # Lire le CSV
        df = pd.read_csv(file_path)

        # Vérifier que la colonne existe
        if "RR_ms" not in df.columns:
            print("❌ Colonne RR_ms introuvable")
            continue

        # Extraire les valeurs
        data_list = df["RR_ms"].dropna().astype(float).tolist()

        if len(data_list) < 10:
            print("❌ Pas assez de données RR")
            continue

        print(f"Nombre d'intervalles RR : {len(data_list)}")

        # Construire le payload JSON
        payload = {
            "type": "RRI",
            "data": data_list,
            "analysis": {
                "type": "timevarying"
            }
        }

        headers = {
            "Authorization": f"Bearer {access_token}",
            "X-Api-Key": api_key,
            "Content-Type": "application/json"
        }

        # Envoyer la requête
        response = requests.post(url_api, headers=headers, json=payload)

        print("Status code:", response.status_code)

        # Vérifier si la réponse est OK
        if response.status_code == 200:
            try:
                result = response.json()
                print("✅ Analyse réussie")
            except:
                print("⚠ Réponse non JSON reçue")
                result = {"error": response.text}
        else:
            print("❌ Erreur API :", response.text)
            result = {"error": response.text}

        # Sauvegarder le résultat
        output_file = output_folder / f"{file_path.stem}_result.json"
        with open(output_file, "w", encoding="utf-8") as out_f:
            json.dump(result, out_f, indent=2)

        # Petite pause pour éviter rate limit
        time.sleep(1)

    except Exception as e:
        print("❌ Erreur générale :", e)

print("\nTraitement terminé ✅")


--- Traitement de 0101EMS_bloc1.csv ---
Nombre d'intervalles RR : 3303
Status code: 200
✅ Analyse réussie

--- Traitement de 0101EMS_bloc2.csv ---
Nombre d'intervalles RR : 2925
Status code: 200
✅ Analyse réussie

--- Traitement de 0101EMS_bloc3.csv ---
Nombre d'intervalles RR : 3201
Status code: 200
✅ Analyse réussie

--- Traitement de 0102EMS_bloc1.csv ---
Nombre d'intervalles RR : 640
Status code: 200
✅ Analyse réussie

--- Traitement de 0102EMS_bloc2.csv ---
Nombre d'intervalles RR : 1845
Status code: 200
✅ Analyse réussie

--- Traitement de 0102EMS_bloc3.csv ---
Nombre d'intervalles RR : 2705
Status code: 200
✅ Analyse réussie

--- Traitement de 0102PCR_bloc1.csv ---
Nombre d'intervalles RR : 1003
Status code: 200
✅ Analyse réussie

--- Traitement de 0102PCR_bloc2.csv ---
Nombre d'intervalles RR : 3683
Status code: 200
✅ Analyse réussie

--- Traitement de 0102PCR_bloc3.csv ---
Nombre d'intervalles RR : 1735
Status code: 200
✅ Analyse réussie

--- Traitement de 0103BPS_bloc1.csv -

In [4]:
json_folder_v4 = Path(r"C:\Users\judupont\Desktop\stat_bb_rr_v4")
df_global_v4_path = r"C:\Users\judupont\Desktop\df_global_v4.txt"

metrics_to_extract = [ "mean_rr", "sdnn", "mean_hr", "min_hr", "max_hr", "rmssd", "pnn50", "lf_power", "hf_power", "lf_hf_power"]

metrics_final_names = ["Mean_RR_ms", "SDNN_ms", "Mean_HR_beats/min", "Min_HR_beats/min", "Max_HR_beats/min", "RMSSD_ms", "pNN50_%", "LF_power_ms2", "HF_power_ms2", "LF_HF_ratio"]

metrics_mapping = dict(zip(metrics_to_extract, metrics_final_names))

# --------------------------
# TRAITEMENT JSON V4
# --------------------------
candidates_v4 = {}

for file_path in json_folder_v4.glob("*.json"):
    with open(file_path, "r", encoding="utf-8") as f:
        try:
            data = json.load(f)
        except:
            continue 
    if "analysis" not in data:
        continue
    analysis = data["analysis"]

    match = re.match(r"(\d{4}[A-Z]{3})_bloc(\d+)_result", file_path.stem)
    if not match:
        continue
    inclusion_number = match.group(1)
    bloc_number = match.group(2)

    if inclusion_number not in candidates_v4:
        candidates_v4[inclusion_number] = {"Numero_inclusion": inclusion_number, "effective_prc_blocs": []}

    for metric in metrics_to_extract:
        key_mean = f"{metrics_mapping[metric]}_bloc{bloc_number}"
        if metric in analysis:
            values = analysis[metric]
            if isinstance(values, list) and len(values) > 0:
                arr = np.array(values, dtype=float)
                arr_clean = arr[~np.isnan(arr)]
                candidates_v4[inclusion_number][key_mean] = np.nanmean(arr_clean) if len(arr_clean) > 0 else np.nan
            else:
                candidates_v4[inclusion_number][key_mean] = values
        else:
            candidates_v4[inclusion_number][key_mean] = np.nan

    if "effective_prc" in analysis:
        arr_eff = np.nan_to_num(np.array(analysis["effective_prc"], dtype=float), nan=0.0)
        candidates_v4[inclusion_number]["effective_prc_blocs"].append(np.mean(arr_eff))

# Calcul global effective_prc
for cand in candidates_v4.values():
    cand["effective_prc"] = np.mean(cand["effective_prc_blocs"]) if len(cand["effective_prc_blocs"]) > 0 else np.nan
    del cand["effective_prc_blocs"]

# DataFrame final
df_results_v4 = pd.DataFrame(list(candidates_v4.values()))
df_results_v4 = df_results_v4.sort_values(by="Numero_inclusion").reset_index(drop=True)

# Merge Condition + Feuille
df_global_v4 = pd.read_csv(df_global_v4_path, sep="\t")
df_global_small_v4 = df_global_v4[["Numero_inclusion", "Feuille"]]
df_global_small_v4 = df_global_small_v4[df_global_small_v4["Numero_inclusion"].isin(df_results_v4["Numero_inclusion"])]
df_results_v4 = df_results_v4.merge(df_global_small_v4, on="Numero_inclusion", how="left")

cols = df_results_v4.columns.tolist()
for col_to_move in ["Feuille"]:
    if col_to_move in cols:
        cols.remove(col_to_move)
        cols.insert(1, col_to_move)
df_results_v4 = df_results_v4[cols]

# Tri décroissant selon effective_prc
df_results_v4 = df_results_v4.sort_values(by="effective_prc", ascending=False).reset_index(drop=True)

In [5]:
# Vérification des NaN pour V4
mask_nan = df_results_v4.isna().any(axis=1)
lignes_nan = df_results_v4[mask_nan]

# Créer la liste des candidats avec NaN
candidats_nan_v4 = lignes_nan['Numero_inclusion'].tolist()

if candidats_nan_v4:
    print("⚠️ Candidats avec au moins un NaN (V4) :")
    for c in candidats_nan_v4:
        print(f"• {c}")
else:
    print("✅ Aucun NaN détecté dans le DataFrame V4.")

# Optionnel : supprimer les lignes avec NaN
df_results_v4 = df_results_v4[~mask_nan].reset_index(drop=True)
print(f"\n🎯 Nombre final de candidats conservés (V4) : {len(df_results_v4)}")

⚠️ Candidats avec au moins un NaN (V4) :
• 0109GSS
• 0323RFS
• 0337BFS
• 0302ZMR
• 0305LCR
• 0333MMS
• 0112DRR
• 0110LPR
• 0408SJS

🎯 Nombre final de candidats conservés (V4) : 89


In [6]:
df_results_v4 = df_results_v4.round(2)

# --------------------------
# Affichage final
# --------------------------

pd.set_option('display.max_columns', None)
df_results_v4

,Numero_inclusion,Feuille,Mean_RR_ms_bloc1,SDNN_ms_bloc1,Mean_HR_beats/min_bloc1,Min_HR_beats/min_bloc1,Max_HR_beats/min_bloc1,RMSSD_ms_bloc1,pNN50_%_bloc1,LF_power_ms2_bloc1,HF_power_ms2_bloc1,LF_HF_ratio_bloc1,Mean_RR_ms_bloc2,SDNN_ms_bloc2,Mean_HR_beats/min_bloc2,Min_HR_beats/min_bloc2,Max_HR_beats/min_bloc2,RMSSD_ms_bloc2,pNN50_%_bloc2,LF_power_ms2_bloc2,HF_power_ms2_bloc2,LF_HF_ratio_bloc2,Mean_RR_ms_bloc3,SDNN_ms_bloc3,Mean_HR_beats/min_bloc3,Min_HR_beats/min_bloc3,Max_HR_beats/min_bloc3,RMSSD_ms_bloc3,pNN50_%_bloc3,LF_power_ms2_bloc3,HF_power_ms2_bloc3,LF_HF_ratio_bloc3,effective_prc
0,0114LMS,APHM,739.08,14.73,81.20,75.80,88.99,7.93,0.01,161.18,24.20,6.52,744.87,14.30,80.58,74.08,89.30,9.60,0.04,130.93,37.96,3.76,760.38,15.13,78.94,72.50,86.69,12.13,0.56,134.56,68.15,2.48,99.98
1,0127CMS,LPC,1001.33,31.26,59.93,55.24,67.68,28.68,8.49,525.28,288.38,1.81,1087.40,32.24,55.19,51.52,61.66,33.40,12.83,533.92,365.12,1.54,1164.62,33.57,51.54,48.23,57.27,37.83,17.62,486.71,448.87,1.16,99.97
2,0112BSR,APHM,848.55,33.12,70.74,64.74,82.80,22.22,3.49,771.28,191.67,4.57,903.56,25.06,66.42,61.99,76.56,22.45,2.02,372.71,201.05,1.85,906.34,25.58,66.24,61.56,76.19,22.39,2.62,386.02,196.57,1.96,99.95
3,0119BIR,LPC,868.56,27.26,69.09,63.83,77.11,18.83,1.64,588.24,112.11,6.47,867.98,24.60,69.15,63.51,77.56,21.72,3.61,388.55,173.88,2.30,906.56,26.24,66.19,61.89,73.82,19.90,2.37,489.29,142.42,3.68,99.95
4,0901SMR,CGD,798.47,29.24,75.38,66.06,87.30,17.51,0.86,684.21,117.24,6.35,856.03,29.64,70.14,63.10,80.48,20.70,2.80,621.87,167.45,4.03,922.07,27.91,65.12,59.37,74.22,22.87,4.29,517.35,177.78,2.99,99.94
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
84,0140LMR,LPC,831.28,14.64,72.19,67.99,76.84,13.86,0.82,110.98,68.93,1.73,896.62,21.10,66.93,62.33,72.68,24.21,6.87,140.70,185.14,1.54,905.77,16.76,66.25,62.87,70.51,17.50,2.32,130.31,74.64,1.82,76.16
85,0904KJS,CGD,726.04,17.30,82.64,78.41,89.08,22.19,9.09,58.16,180.58,1.41,753.17,19.26,79.68,74.08,87.16,21.89,6.49,117.92,148.97,1.33,789.82,25.75,76.00,71.30,85.08,33.29,13.93,174.98,311.07,0.80,72.38
86,0403DCR,POITIERS,675.08,23.38,88.88,79.43,96.74,12.72,0.04,367.15,56.74,7.08,700.08,20.57,85.72,76.71,92.65,12.67,0.28,286.28,77.44,3.62,726.53,20.49,82.60,75.12,89.93,14.22,0.30,179.65,69.45,2.49,64.82
87,0102PCR,APHM,725.79,37.04,82.74,70.24,96.56,40.77,17.66,403.96,335.51,1.74,835.16,64.12,71.88,62.27,88.69,65.20,35.19,1548.04,835.25,2.74,894.34,65.82,67.13,56.36,83.56,56.18,27.57,2490.27,914.65,3.04,55.63


## Enregistrement 

In [8]:
df_results_v4.to_csv(r"C:\Users\judupont\Desktop\HRV_RtoR_V4_kubios.csv", index=False)
df_results_v2.to_csv(r"C:\Users\judupont\Desktop\HRV_RtoR_V2_kubios.csv", index=False,encoding="utf-8-sig")